In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re
import os

# Verificar que los archivos están disponibles
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/mateotorrest/sivigila-divipola-2021/DA-SIVIGILA_2021_INS_20251018 (2).csv
/kaggle/input/datasets/mateotorrest/sivigila-divipola-2021/DIVIPOLA_CentrosPoblados.csv


In [2]:
ruta_sivigila = "/kaggle/input/datasets/mateotorrest/sivigila-divipola-2021/DA-SIVIGILA_2021_INS_20251018 (2).csv"

df_SIVIGILA = pd.read_csv(ruta_sivigila, encoding="latin-1", low_memory=False)

print("Shape:", df_SIVIGILA.shape)
print("Columnas:", df_SIVIGILA.columns.tolist())
df_SIVIGILA.head()

Shape: (187598, 7)
Columnas: ['COD_EVE', 'Nombre_evento', 'SEMANA', 'ANO', 'Municipio_ocurrencia', 'Departamento_ocurrencia', 'conteo']


,COD_EVE,Nombre_evento,SEMANA,ANO,Municipio_ocurrencia,Departamento_ocurrencia,conteo
0,100,ACCIDENTE OFIDICO,1,"2,021",AMALFI,ANTIOQUIA,2
1,100,ACCIDENTE OFIDICO,1,"2,021",BOSCONIA,CESAR,1
2,100,ACCIDENTE OFIDICO,1,"2,021",EL TAMBO,CAUCA,1
3,100,ACCIDENTE OFIDICO,1,"2,021",GAMARRA,CESAR,1
4,100,ACCIDENTE OFIDICO,1,"2,021",MARIA LA BAJA,BOLIVAR,2


In [3]:
ruta_divipola = "/kaggle/input/datasets/mateotorrest/sivigila-divipola-2021/DIVIPOLA_CentrosPoblados.csv"

df_divipola = pd.read_csv(ruta_divipola, encoding="latin-1", sep=";")

print("Shape:", df_divipola.shape)
print("Columnas:", df_divipola.columns.tolist())
df_divipola.head()

Shape: (8421, 9)
Columnas: ['Código_Departamento', 'Nombre_Departamento', 'Código_Municipio', 'Nombre_Municipio', 'Código_Entidad', 'Nombre_Entidad', 'Tipo', 'Longitud', 'Latitud']


,Código_Departamento,Nombre_Departamento,Código_Municipio,Nombre_Municipio,Código_Entidad,Nombre_Entidad,Tipo,Longitud,Latitud
0,5,ANTIOQUIA,5001,MEDELLÍN,5001000,"MEDELLÍN, DISTRITO ESPECIAL DE CIENCIA, TECNOL...",CM,"-75,581775","6,246631"
1,5,ANTIOQUIA,5001,MEDELLÍN,5001001,PALMITAS,CP,"-75,690573","6,343919"
2,5,ANTIOQUIA,5001,MEDELLÍN,5001004,SANTA ELENA,CP,"-75,501293","6,210599"
3,5,ANTIOQUIA,5001,MEDELLÍN,5001009,ALTAVISTA,CP,"-75,643706","6,221429"
4,5,ANTIOQUIA,5001,MEDELLÍN,5001010,AGUAS FRÍAS,CP,"-75,633948","6,233335"


In [4]:
# Filtrar solo cabeceras municipales (tipo CM)
df_divipola_muni = df_divipola[df_divipola['Tipo'] == 'CM'].copy()

# Quedarnos solo con las columnas que necesitamos
df_divipola_muni = df_divipola_muni[[
    'Código_Departamento', 'Nombre_Departamento',
    'Código_Municipio', 'Nombre_Municipio'
]].copy()

# Renombrar columnas para que sean más fáciles de usar
df_divipola_muni = df_divipola_muni.rename(columns={
    'Código_Departamento': 'doc_dep',
    'Nombre_Departamento': 'departamento',
    'Código_Municipio':    'cod_muni',
    'Nombre_Municipio':    'municipio'
})

print("Shape DIVIPOLA filtrado:", df_divipola_muni.shape)
df_divipola_muni.head()

Shape DIVIPOLA filtrado: (1104, 4)


,doc_dep,departamento,cod_muni,municipio
0,5,ANTIOQUIA,5001,MEDELLÍN
28,5,ANTIOQUIA,5002,ABEJORRAL
32,5,ANTIOQUIA,5004,ABRIAQUÍ
34,5,ANTIOQUIA,5021,ALEJANDRÍA
35,5,ANTIOQUIA,5030,AMAGÁ


In [5]:
import unicodedata
import re

def normalizar(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'\s+', ' ', texto)
    return texto

print("✅ Función lista")

✅ Función lista


In [6]:
# --- Limpiar SIVIGILA ---
# Corregir columna ANO (venía como "2,021")
df_SIVIGILA['ANO'] = (df_SIVIGILA['ANO']
                      .astype(str)
                      .str.replace(',', '')
                      .str.extract(r'(\d{4})')[0]
                      .astype(int))

# Normalizar nombres de territorio
df_SIVIGILA['Departamento_ocurrencia'] = df_SIVIGILA['Departamento_ocurrencia'].apply(normalizar)
df_SIVIGILA['Municipio_ocurrencia']    = df_SIVIGILA['Municipio_ocurrencia'].apply(normalizar)

# --- Limpiar DIVIPOLA ---
df_divipola_muni['departamento'] = df_divipola_muni['departamento'].apply(normalizar)
df_divipola_muni['municipio']    = df_divipola_muni['municipio'].apply(normalizar)

print("✅ Normalización lista")
print("\nEjemplo SIVIGILA:")
print(df_SIVIGILA[['Departamento_ocurrencia', 'Municipio_ocurrencia']].head(3))
print("\nEjemplo DIVIPOLA:")
print(df_divipola_muni[['departamento', 'municipio']].head(3))

✅ Normalización lista

Ejemplo SIVIGILA:
  Departamento_ocurrencia Municipio_ocurrencia
0               ANTIOQUIA               AMALFI
1                   CESAR             BOSCONIA
2                   CAUCA             EL TAMBO

Ejemplo DIVIPOLA:
   departamento  municipio
0     ANTIOQUIA   MEDELLIN
28    ANTIOQUIA  ABEJORRAL
32    ANTIOQUIA   ABRIAQUI


In [7]:
df_sivigila_divipola = df_SIVIGILA.merge(
    df_divipola_muni[['departamento', 'municipio', 'cod_muni', 'doc_dep']],
    left_on  = ['Departamento_ocurrencia', 'Municipio_ocurrencia'],
    right_on = ['departamento', 'municipio'],
    how      = 'left'
)

# Ver cuántos cruzaron bien
total     = len(df_sivigila_divipola)
con_match = df_sivigila_divipola['cod_muni'].notna().sum()
sin_match = df_sivigila_divipola['cod_muni'].isna().sum()

print(f"Total registros:         {total:,}")
print(f"Con cruce DIVIPOLA:      {con_match:,} ({con_match/total*100:.1f}%)")
print(f"Sin cruce (descartados): {sin_match:,} ({sin_match/total*100:.1f}%)")

Total registros:         187,598
Con cruce DIVIPOLA:      138,770 (74.0%)
Sin cruce (descartados): 48,828 (26.0%)


In [8]:
df_SIVIGILA_limpio = df_sivigila_divipola.dropna(subset=['cod_muni']).copy()

# Dejar solo columnas relevantes
cols_finales = [
    'COD_EVE', 'Nombre_evento', 'SEMANA', 'ANO',
    'Municipio_ocurrencia', 'Departamento_ocurrencia',
    'conteo', 'doc_dep', 'departamento', 'cod_muni', 'municipio'
]
df_SIVIGILA_limpio = df_SIVIGILA_limpio[cols_finales]

print("Shape final:", df_SIVIGILA_limpio.shape)
df_SIVIGILA_limpio.head()

Shape final: (138770, 11)


,COD_EVE,Nombre_evento,SEMANA,ANO,Municipio_ocurrencia,Departamento_ocurrencia,conteo,doc_dep,departamento,cod_muni,municipio
0,100,ACCIDENTE OFIDICO,1,2021,AMALFI,ANTIOQUIA,2,5.0,ANTIOQUIA,5031.0,AMALFI
1,100,ACCIDENTE OFIDICO,1,2021,BOSCONIA,CESAR,1,20.0,CESAR,20060.0,BOSCONIA
2,100,ACCIDENTE OFIDICO,1,2021,EL TAMBO,CAUCA,1,19.0,CAUCA,19256.0,EL TAMBO
3,100,ACCIDENTE OFIDICO,1,2021,GAMARRA,CESAR,1,20.0,CESAR,20295.0,GAMARRA
4,100,ACCIDENTE OFIDICO,1,2021,MARIA LA BAJA,BOLIVAR,2,13.0,BOLIVAR,13442.0,MARIA LA BAJA


In [9]:
df_SIVIGILA_limpio.to_csv("/kaggle/working/df_SIVIGILA_limpio.csv", index=False)

print("✅ Archivo guardado en /kaggle/working/")
print(f"Filas: {len(df_SIVIGILA_limpio):,}")
print(f"Columnas: {df_SIVIGILA_limpio.columns.tolist()}")

✅ Archivo guardado en /kaggle/working/
Filas: 138,770
Columnas: ['COD_EVE', 'Nombre_evento', 'SEMANA', 'ANO', 'Municipio_ocurrencia', 'Departamento_ocurrencia', 'conteo', 'doc_dep', 'departamento', 'cod_muni', 'municipio']
